# Question 8
Design a data-sharing decision matrix for Cyntexa: for a given partner scenario (has their own
Databricks workspace vs. doesn't; needs tables only vs. needs AI assets), determine which
OpenSharing protocol — Databricks-to-Databricks vs. Databricks-to-Open — applies and why.

# Cyntexa OpenSharing & Delta Sharing Protocol Matrix

## 📌 Overview
This document serves as the governing standard for data-sharing architectures at Cyntexa. Depending on the partner's infrastructure and the assets to be shared, engineers must follow the defined decision matrix to implement either **Databricks-to-Databricks Sharing** or **Databricks-to-Open Sharing**.

---

## 🚦 Decision Matrix

| Scenario | Partner Platform | Required Asset Type | Applicable Protocol | Primary Reason |
| :--- | :--- | :--- | :--- | :--- |
| **1** | Databricks (UC Enabled) | Tables / Data | **Databricks-to-Databricks** | Direct metastore alignment without credential/token rotation overhead. |
| **2** | Databricks (UC Enabled) | AI Models, Volumes, Notebooks | **Databricks-to-Databricks** | Unity Catalog AI assets are strictly restricted to native metastore sharing. |
| **3** | Non-Databricks (PowerBI, Snowflake, etc.) | Tables / Data | **Databricks-to-Open** | Enables secure token-based reading using open Delta Sharing connectors. |
| **4** | Non-Databricks (PowerBI, Snowflake, etc.) | AI Models, Volumes, Notebooks | **Unsupported Directly** | Protocol Limitation: Open sharing does not support ML/Volume governance. |

---

## 💡 Key Architectural Rules

1. **Native Unity Catalog Governance (Databricks-to-Databricks)**
   * Preferred for all partners operating on Databricks.
   * Requires the recipient’s **Metastore ID** (`<cloud>:<region>:<metastore-guid>`).
   * No static access keys or credential files needed.

2. **Open Delta Sharing (Databricks-to-Open)**
   * Mandatory for external engines (Python/Pandas, Spark, Power BI, Excel, Snowflake).
   * Generates a downloadable `.share` credential file or OIDC bearer token link.
   * Enforce short expiry cycles (30–90 days max) on generated tokens.

3. **AI Models & Volume Restrictions**
   * Non-tabular objects (UC AI Models, Volumes, Notebooks) **cannot** be exported over Open Sharing.
   * **Workaround for Non-Databricks Partners (Scenario 4):**
     * Deploy the AI Model via **Databricks Model Serving** as a REST API endpoint.
     * Share pre-computed model outputs via standard open shared tables.

---

## 💻 Quickstart Setup Commands (Databricks SQL)

### 1. Create a Share
```sql
CREATE SHARE partner_data_share;

-- Add tabular assets
ALTER SHARE partner_data_share ADD TABLE main.analytics.sales_summary;

-- Add AI model (Supported ONLY for Databricks-to-Databricks recipients)
ALTER SHARE partner_data_share ADD MODEL main.ml_models.churn_predictor;

# Question 9

Evaluate query federation vs. building a nightly copy pipeline for the Postgres source: under what
data-freshness and query-volume conditions does federation stop making sense?

An architectural comparison and decision framework for integrating PostgreSQL with Databricks using **Query Federation (Lakehouse Federation)** versus a **Nightly ETL/ELT Batch Pipeline** (e.g., Lakeflow Pipelines, Auto Loader, or Delta Live Tables).

---

## Architectural Comparison

```
[ Query Federation Protocol ]
Databricks (SQL Warehouse)  ─── Pushdown Query ───►  PostgreSQL (OLTP Engine)
                           ◄── Stream Raw Rows ───

[ Nightly Batch ETL Pipeline ]
PostgreSQL (OLTP Engine) ─── Raat ko Batch Extract ───► Bronze/Silver Delta Tables (Parquet/UC)
                                                          │
                                                          ▼
Databricks Queries ──────────────────────────────────────► Local Delta Lake Storage

```

| Metric / Dimension | Query Federation (Databricks Foreign Catalog) | Nightly Batch Copy Pipeline |
| --- | --- | --- |
| **Data Storage Location** | Remains inside PostgreSQL OLTP disk. | Written as optimized Delta Parquet files in ADLS/S3/GCS. |
| **Compute Overhead** | Offloaded primarily to PostgreSQL CPU/RAM via query pushdown. | Offloaded to Databricks Spark clusters during batch run. |
| **Data Freshness** | Real-time (Zero Latency). | Stale (T+1 Day / 24-hour lag). |
| **OLTP Database Impact** | **High Risk:** Analytical scans directly affect app production DB. | **Isolated:** Single nightly read/replica scan or CDC read. |
| **Query Performance** | Limited by Postgres network throughput and single-node RAM/CPU. | **High Speed:** Photon engine, Z-Ordering, Data Skipping, liquid clustering. |
| **Cost Dynamics** | Low storage, high Postgres compute costs (requires vertical scaling). | Cheap object storage, burst Spark compute during batch windows. |

---

## Architectural Trade-offs

### 1. Query Federation

* **Pros:**
* **Zero ETL Overhead:** No DAGs, orchestration pipelines, or schemas to sync.
* **Zero Latency:** Queries immediately reflect operational data written a second ago.
* **Storage Savings:** Prevents duplication of terabytes of data across environments.


* **Cons:**
* **OLTP Resource Contention:** Full table scans or unindexed joins on Postgres can saturate connection pools, spike CPU, and lock rows—impacting core production applications.
* **No Delta Lake Optimizations:** Loses out on Databricks performance features like Liquid Clustering, Z-Ordering, Caching, and Parquet columnar layouts.
* **Network Bottlenecks:** Transferring large record sets over JDBC/ODBC bridges creates significant network latency.



### 2. Nightly ETL Pipeline

* **Pros:**
* **Complete OLTP Isolation:** Production databases remain unaffected by downstream analytics workloads.
* **Maximum Analytical Performance:** Queries execute directly on columnar Delta tables using Spark's vectorization and C++ Photon engine.
* **Historical Versioning:** Enables time travel (`RESTORE`, `VERSION AS OF`) and Change Data Capture (SCD Type 1 & 2 tracking).


* **Cons:**
* **Data Stale for 24 Hours:** Not suitable for real-time operational dashboards or live alerting.
* **Pipeline Maintenance Overhead:** Requires schema drift handling, failure alerting, retries, and data validation steps.



---

## Threshold Evaluation Framework

Query Federation stops making sense when operational constraints cross specific performance, cost, and infrastructure boundaries:

```
                          ┌───────────────────────────┐
                          │   Data Freshness Needed?  │
                          └─────────────┬─────────────┘
                                        │
                         ┌──────────────┴──────────────┐
                   Sub-second / Live              24 Hours OK
                         │                             │
                         ▼                             ▼
              ┌─────────────────────┐       ┌─────────────────────┐
              │  Query Federation   │       │ Nightly ETL Copy    │
              │  or CDC Streaming    │       └─────────────────────┘
              └──────────┬──────────┘
                         │
        ┌────────────────┴────────────────┐
        │  Check Query Volume & Postgres  │
        │           System Load           │
        └────────────────┬────────────────┘
                         │
     Rows Scanned > 10M OR Queries > 100/hr
                         │
                         ▼
        ┌─────────────────────────────────┐
        │ Federation FAILS:               │
        │ Shift to CDC / Batch Pipeline   │
        └─────────────────────────────────┘

```

### When Does Federation Stop Making Sense?

1. **Query Volume Thresholds:**
* **Result Set Size:** When individual federated queries fetch or process **more than 10 million rows**. Converting row-oriented Postgres data to Spark columnar format over JDBC degrades throughput.
* **Concurrent Analytical Load:** When more than **50–100 analytical queries per hour** hit the federated source. This consumes execution threads and connection pools on PostgreSQL.


2. **Data Freshness Requirements:**
* When business users accept a **24-hour SLA (T+1)**, Query Federation provides no architectural benefit and introduces unnecessary production risk.


3. **Complex Aggregations & Multi-Table Joins:**
* If Unity Catalog cannot push down complex filter predicate filters, window functions, or multi-way joins to PostgreSQL, Databricks must pull the entire raw table over JDBC to run the execution plan locally.


4. **Production DB Health Constraints:**
* If PostgreSQL CPU utilization exceeds **60% under normal OLTP traffic**, adding federated analytical queries risks cascading system outages.



---

## Final Recommendation Rule

* **Use Query Federation for:** Low-frequency ad-hoc exploratory queries, dimensional lookup table joining (e.g., joining an operational user table to a massive Delta fact table), or scenarios requiring live operational audit data.
* **Migrate to ETL (Batch/CDC) when:** Analytics workloads require full-table scans, daily reporting, heavy dashboard usage, or when business decisions tolerate daily updates.

# Question 10

Propose an LTAP architecture for a new Cyntexa feature (e.g., a real-time inventory-check app) that
needs both OLTP writes (Lakebase) and OLAP analytics (Lakehouse) on the same data, specifying
what syncs where and who owns each side operationally.



---

### Architectural Overview

```
 [ Mobile / Web App ]
          │
          │ (1. Low-latency OLTP Writes / Reads)
          ▼
 ┌───────────────────────────────────┐
 │ Lakebase (OLTP Operational DB)    │ ◄── Managed by Backend / Product Eng
 │ - Primary Engine: PostgreSQL      │
 │ - Row-based Storage               │
 └─────────────────┬─────────────────┘
                   │
                   │ (2. Change Data Capture via Lakeflow Connect / Debezium)
                   ▼
 ┌───────────────────────────────────┐
 │ Databricks Lakehouse (OLAP Engine)│ ◄── Managed by Data Engineering / Analytics
 │ - Storage: Medallion (Delta Lake) │
 │ - Compute: Serverless / Photon    │
 └───────────────────────────────────┘

```

---

### Data Sync Strategy & Pipeline Mechanics

1. **Transactional Writes (OLTP - Lakebase)**
* Every inventory update, purchase event, or stock reservation hits **Lakebase** directly via standard SQL/ORMs.
* Row-level locking and ACID guarantees ensure zero race conditions during stock updates.


2. **Real-Time Data Replication (CDC to Lakehouse)**
* **Mechanism:** **Lakeflow Connect** (or Debezium + Apache Kafka) reads the PostgreSQL write-ahead log (WAL) from Lakebase in near real-time.
* **Bronze Layer:** Continuous ingestion of raw WAL events into a streaming Bronze Delta table.
* **Silver Layer:** Delta Live Tables (DLT) or Structured Streaming apply `APPLY CHANGES INTO` (SCD Type 1 merge) to maintain an exact, up-to-date replica of the operational state within seconds (sub-minute latency).
* **Gold Layer:** Aggregated views (e.g., store-level stock prediction, demand forecasting, regional depletion rates) for BI dashboards and ML models.



---

### Operational Ownership Matrix

| Area | Operational Component | Owning Team | Responsibilities |
| --- | --- | --- | --- |
| **OLTP Side** | **Lakebase (Operational DB)** | **Backend / Product Software Engineers** | Schema migrations, connection pooling, indexing strategy for app queries, meeting app SLAs (<50ms response times). |
| **Data Bridge** | **CDC Ingestion Pipeline** | **Joint Ownership (Data + Platform)** | WAL replication health, stream offset monitoring, schema drift detection, network throughput optimization. |
| **OLAP Side** | **Databricks Lakehouse** | **Data Engineering / Analytics Team** | DLT pipeline health, Gold layer transformations, Z-Ordering / Liquid Clustering, ML model features, Unity Catalog access policies. |

---

### Why This Architecture Works for Cyntexa

* **Zero OLTP Impact:** Operational app updates remain extremely fast because complex analytical queries never touch the operational Lakebase instance.
* **Unified Governance:** Unity Catalog enforces security and access control policies consistently across both operational metadata and analytical Delta tables.
* **Near Real-Time Insights:** The CDC streaming loop delivers near real-time analytics to business stakeholders without performance compromises.